# Milestone 3: Analisis Temporal dan Insights APBD Indonesia

**Mata kuliah:** Analisis Big Data  
**Nama:** Afzal Raditya Dharma  
**NIM:** 202310370311236  
**GitHub:** AfzalRaditya

Fokus analisis: temporal. Tujuan: menilai tren APBD tahunan, perubahan distribusi
penyerapan, pengaruh pencilan dari tahun ke tahun, pola snapshot bulanan, serta
perbandingan September pada tahun lengkap dan tahun parsial. Pemda digunakan
sebagai unit panel agar perbandingan waktu konsisten; tidak dibuat peringkat wilayah.
Sumber analisis adalah hasil notebook cleaning, bukan data mentah.

## 1. Memuat dan memvalidasi data

In [1]:
from pathlib import Path
import duckdb
import polars as pl
import plotly.express as px
import plotly.io as pio
from IPython.display import display, Markdown

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'data/processed/apbd_indonesia_bersih.parquet').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Jalankan notebook 02 terlebih dahulu untuk menghasilkan data bersih.')
DATA_PATH = ROOT / 'data/processed/apbd_indonesia_bersih.parquet'
FIG_DIR = ROOT / 'output/figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
pio.templates.default = 'plotly_white'
lf = pl.scan_parquet(DATA_PATH)
print(f"Jumlah baris: {lf.select(pl.len()).collect().item():,}")
display(lf.head().collect())

def show(fig, filename):
    fig.update_layout(hovermode='closest', margin=dict(t=90))
    fig.write_html(FIG_DIR / filename, include_plotlyjs=True)
    fig.show()

Jumlah baris: 3,132,863


tahun,bulan,id_provinsi,provinsi,id_pemda,pemda,no_akun,akun,anggaran,realisasi,persentase,persentase_sumber,flag_anggaran_negatif,flag_realisasi_negatif,flag_rasio_tidak_terdefinisi,flag_rasio_ekstrem
i16,i8,str,str,str,str,i16,str,f64,f64,f64,f64,bool,bool,bool,bool
2011,1,"""01""","""Provinsi Aceh""","""00""","""Prov. Aceh""",1,"""Pendapatan""",7.0894e12,7.6093e12,107.333875,107.33,false,false,false,false
2011,1,"""01""","""Provinsi Aceh""","""00""","""Prov. Aceh""",2,"""PAD""",7.9728e11,8.0284e11,100.696324,100.7,false,false,false,false
2011,1,"""01""","""Provinsi Aceh""","""00""","""Prov. Aceh""",3,"""Pajak Daerah""",6.2271e11,5.8618e11,94.134568,94.13,false,false,false,false
2011,1,"""01""","""Provinsi Aceh""","""00""","""Prov. Aceh""",4,"""Retribusi Daerah""",1.3264e10,9.4160e9,70.988422,70.99,false,false,false,false
2011,1,"""01""","""Provinsi Aceh""","""00""","""Prov. Aceh""",5,"""Hasil Pengelolaan Kekayaan Dae…",1.0200e11,6.6007e10,64.712874,64.71,false,false,false,false


### Aturan agregasi dan batas interpretasi

- Akun bersifat hierarkis: akun induk dan anak tidak dijumlahkan bersama.
  Nomor akun memiliki lebih dari satu nama dalam data; pemilihan menggunakan nama akun.
- Pendapatan/Pendapatan Daerah dan Belanja/Belanja Daerah diperlakukan sebagai alias
  akun total. Keunikan per pemda–tahun–bulan–kategori diperiksa sebelum agregasi.
- Nilai bulan diperlakukan sebagai **snapshot laporan**, tidak dijumlahkan menjadi
  total tahunan. Untuk perbandingan tahunan digunakan Desember dan panel pemda tetap.
- Rasio dihitung ulang sebagai 100 × realisasi/anggaran; anggaran nol menghasilkan
  null. Nominal sumber dipertahankan; flag cleaning membantu meninjau rasio ekstrem.
- Nilai adalah rupiah nominal, tanpa koreksi inflasi atau penduduk. Penjumlahan
  pemda provinsi dan kabupaten/kota bukan konsolidasi fiskal nasional karena transfer
  antar-pemda dapat tercatat di kedua sisi. ID pemda unik memakai pasangan provinsi–pemda.
- Data 2026 hanya sampai September. Keberadaan 12 bulan belum membuktikan adanya
  12 pembaruan: laporan lama dapat diulang. Pola bulanan perlu diaudit sebelum
  ditafsirkan sebagai musim pencairan belanja.

In [2]:
aliases = {'Pendapatan': 'Pendapatan', 'Pendapatan Daerah': 'Pendapatan',
           'Belanja': 'Belanja', 'Belanja Daerah': 'Belanja'}
keys = ['tahun', 'bulan', 'id_provinsi', 'id_pemda', 'kategori']
base = (lf.filter(pl.col('akun').is_in(list(aliases)))
        .with_columns(pl.col('akun').replace_strict(aliases).alias('kategori'))
        .select(*keys, 'provinsi', 'pemda', 'anggaran', 'realisasi').collect())
assert base.height > 0
assert base.select(keys).n_unique() == base.height, 'Alias akun ganda: tinjau sebelum menjumlahkan.'
assert base.filter(~pl.col('bulan').is_between(1, 12)).is_empty()
base = base.with_columns(
    pl.when(pl.col('anggaran') > 0)
      .then(100 * pl.col('realisasi') / pl.col('anggaran')).alias('rasio'),
    pl.concat_str('id_provinsi', 'id_pemda', separator=':').alias('wilayah'))
coverage = base.group_by('tahun').agg(
    pl.col('bulan').n_unique().alias('jumlah_bulan'),
    pl.col('bulan').max().alias('bulan_terakhir'),
    pl.col('wilayah').n_unique().alias('jumlah_pemda')).sort('tahun')
display(coverage)


years = coverage.filter(pl.col('bulan_terakhir') == 12)['tahun'].to_list()
dec = base.filter(pl.col('tahun').is_in(years) & (pl.col('bulan') == 12))
panel_ids = dec.group_by('wilayah').len().filter(pl.col('len') == 2 * len(years))['wilayah']
annual = dec.filter(pl.col('wilayah').is_in(panel_ids.implode()))
assert annual.height > 0, 'Panel tetap kosong; periksa kelengkapan data.'
latest = max(years)
snapshot = dec.filter(pl.col('tahun') == latest)
spend = snapshot.filter(pl.col('kategori') == 'Belanja')
print(f'Panel tren: {len(panel_ids)} pemda; snapshot distribusi: Desember {latest}.')

Panel tren: 498 pemda; snapshot distribusi: Desember 2025.


tahun,jumlah_bulan,bulan_terakhir,jumlah_pemda
i16,u32,i8,u32
2011,12,12,498
2012,12,12,498
2013,12,12,498
2014,12,12,513
2015,12,12,516
…,…,…,…
2022,12,12,516
2023,12,12,546
2024,12,12,546


## 2. Analisis temporal dengan DuckDB window function
LAG membandingkan tahun berurutan pada panel tetap. Tahun parsial dikeluarkan.

In [3]:
con = duckdb.connect()
con.register('annual', annual.to_arrow())
result = con.execute("""
    WITH totals AS (
        SELECT tahun, kategori, SUM(anggaran) AS anggaran,
               SUM(realisasi) AS realisasi, COUNT(*) AS jumlah_pemda
        FROM annual GROUP BY tahun, kategori
    )
    SELECT *, 100 * (realisasi / NULLIF(LAG(realisasi) OVER
        (PARTITION BY kategori ORDER BY tahun), 0) - 1) AS pertumbuhan_yoy
    FROM totals ORDER BY kategori, tahun
""")
trend = pl.from_arrow(result.to_arrow_table() if hasattr(result, 'to_arrow_table')
                     else result.fetch_arrow_table())
display(trend)
con.close()

<ipython-input-3-2c415270aeea>:13: UserWarning: Arrow output on a DB-API result makes an extra copy of the rows, use con.sql(query).to_arrow_table() instead.
  trend = pl.from_arrow(result.to_arrow_table() if hasattr(result, 'to_arrow_table')


tahun,kategori,anggaran,realisasi,jumlah_pemda,pertumbuhan_yoy
i16,str,f64,f64,i64,f64
2011,"""Belanja""",4.9772e14,5.0263e14,498,null
2012,"""Belanja""",5.9850e14,5.9964e14,498,19.300261
2013,"""Belanja""",7.1592e14,6.9383e14,498,15.707242
2014,"""Belanja""",8.2502e14,7.6879e14,498,10.803557
2015,"""Belanja""",9.1596e14,8.7378e14,498,13.656522
…,…,…,…,…,…
2021,"""Pendapatan""",1.1050e15,1.1172e15,498,4.741752
2022,"""Pendapatan""",1.0853e15,1.1335e15,498,1.457283
2023,"""Pendapatan""",1.1619e15,1.1182e15,498,-1.349798


## 3. Enam visualisasi interaktif
Grafik juga disimpan sebagai HTML mandiri di `output/figures/`.

In [4]:

fig1 = px.line(trend.with_columns((pl.col('realisasi') / 1e12).alias('triliun')),
    x='tahun', y='triliun', color='kategori', markers=True,
    hover_data=['jumlah_pemda', 'pertumbuhan_yoy'],
    title='1. Realisasi Desember pada panel pemda tetap',
    labels={'triliun': 'Realisasi (Rp triliun)', 'tahun': 'Tahun'})
fig1.update_xaxes(rangeslider_visible=True, dtick=1)
show(fig1, '01_tren_tahunan.html')

In [5]:
fig2 = px.box(annual.filter(pl.col('rasio').is_not_null())
    .with_columns(pl.col('tahun').cast(pl.String)), x='tahun', y='rasio', color='kategori',
    points='outliers', hover_data=['pemda'],
    title='2. Perubahan distribusi rasio Desember pada panel tetap',
    labels={'rasio': 'Realisasi / anggaran (%)', 'tahun': 'Tahun', 'kategori': 'Akun total'})
show(fig2, '02_distribusi_penyerapan.html')

In [6]:
annual_spend = annual.filter(pl.col('kategori') == 'Belanja')
positive = annual_spend.filter((pl.col('anggaran') > 0) & (pl.col('realisasi') > 0))
corr_all = positive.group_by('tahun').agg(
    pl.corr('anggaran', 'realisasi').alias('korelasi'), pl.len().alias('jumlah_pemda')
).with_columns(pl.lit('Semua nominal positif').alias('skenario'))
corr_checked = positive.filter(pl.col('rasio') <= 200).group_by('tahun').agg(
    pl.corr('anggaran', 'realisasi').alias('korelasi'), pl.len().alias('jumlah_pemda')
).with_columns(pl.lit('Rasio <=200% (sensitivitas)').alias('skenario'))
corr_year = pl.concat([corr_all, corr_checked]).sort('skenario', 'tahun')
display(corr_year)
fig3 = px.line(corr_year, x='tahun', y='korelasi', color='skenario', markers=True,
    hover_data=['jumlah_pemda'], title='3. Korelasi anggaran–realisasi belanja dari tahun ke tahun',
    labels={'korelasi': 'Korelasi Pearson', 'tahun': 'Tahun', 'skenario': 'Skenario'})
fig3.update_yaxes(range=[-1.05, 1.05])
show(fig3, '03_anggaran_realisasi.html')

tahun,korelasi,jumlah_pemda,skenario
i16,f64,u32,str
2011,0.993676,497,"""Rasio <=200% (sensitivitas)"""
2012,0.993528,497,"""Rasio <=200% (sensitivitas)"""
2013,0.992584,498,"""Rasio <=200% (sensitivitas)"""
2014,0.962847,497,"""Rasio <=200% (sensitivitas)"""
2015,0.980303,498,"""Rasio <=200% (sensitivitas)"""
…,…,…,…
2021,0.995104,498,"""Semua nominal positif"""
2022,0.993292,498,"""Semua nominal positif"""
2023,0.993678,498,"""Semua nominal positif"""


In [7]:
fig4 = px.bar(trend.with_columns((pl.col('anggaran') / 1e12).alias('triliun')),
    x='tahun', y='triliun', color='kategori', barmode='group',
    hover_data=['jumlah_pemda'], title='4. Perbandingan anggaran pendapatan dan belanja tiap tahun',
    labels={'triliun': 'Anggaran (Rp triliun)', 'tahun': 'Tahun', 'kategori': 'Akun total'})
show(fig4, '04_anggaran_tahunan.html')

In [8]:

monthly = base.filter((pl.col('kategori') == 'Belanja') & pl.col('tahun').is_in(years))
complete = monthly.group_by('wilayah', 'tahun').len().filter(pl.col('len') == 12)
stable_ids = complete.group_by('wilayah').len().filter(pl.col('len') == len(years))['wilayah']
monthly = monthly.filter(pl.col('wilayah').is_in(stable_ids.implode()))
season = monthly.group_by('tahun', 'bulan').agg(pl.col('anggaran').sum(), pl.col('realisasi').sum())
season = season.with_columns((100 * pl.col('realisasi') / pl.col('anggaran')).alias('rasio'))
matrix = season.pivot(on='bulan', index='tahun', values='rasio').sort('tahun')
fig5 = px.imshow(matrix.select([str(i) for i in range(1, 13)]).to_numpy(),
    x=list(range(1, 13)), y=matrix['tahun'].to_list(), aspect='auto',
    labels={'x': 'Bulan laporan', 'y': 'Tahun', 'color': 'Rasio (%)'},
    title=f'5. Rasio snapshot bulanan pada panel tetap ({len(stable_ids)} pemda)')
show(fig5, '05_pola_bulanan.html')


audit = monthly.group_by('tahun', 'wilayah').agg(pl.col('realisasi').n_unique().alias('nilai_unik'))
audit_year = audit.group_by('tahun').agg(
    (100 * (pl.col('nilai_unik') == 1).mean()).alias('persen_pemda_identik')).sort('tahun')
display(audit_year)

tahun,persen_pemda_identik
i16,f64
2011,100.0
2012,100.0
2013,100.0
2014,100.0
2015,100.0
…,…
2021,100.0
2022,100.0
2023,0.0


In [9]:
compare_years = sorted(base.filter(pl.col('bulan') == 9)['tahun'].unique().to_list())[-4:]
september = base.filter((pl.col('bulan') == 9) & (pl.col('kategori') == 'Belanja')
                       & pl.col('tahun').is_in(compare_years))
september_ids = september.group_by('wilayah').len().filter(pl.col('len') == len(compare_years))['wilayah']
september = september.filter(pl.col('wilayah').is_in(september_ids.implode()))
assert september.height > 0, 'Panel September kosong.'
sept_summary = september.group_by('tahun').agg(
    pl.col('rasio').median().alias('median'),
    pl.col('rasio').quantile(.25).alias('q1'),
    pl.col('rasio').quantile(.75).alias('q3'),
    pl.col('rasio').count().alias('jumlah_rasio_valid'),
    pl.len().alias('jumlah_pemda'),
).sort('tahun').with_columns(
    (pl.col('q3') - pl.col('median')).alias('error_atas'),
    (pl.col('median') - pl.col('q1')).alias('error_bawah'))
display(sept_summary)
fig6 = px.line(sept_summary, x='tahun', y='median', markers=True,
    error_y='error_atas', error_y_minus='error_bawah',
    hover_data=['jumlah_pemda', 'jumlah_rasio_valid', 'q1', 'q3'],
    title='6. Penyerapan September pada panel tetap; garis error menunjukkan kuartil',
    labels={'median': 'Median rasio belanja (%)', 'tahun': 'Tahun'})
fig6.update_xaxes(dtick=1)
show(fig6, '06_perbandingan_september.html')

tahun,median,q1,q3,jumlah_rasio_valid,jumlah_pemda,error_atas,error_bawah
i16,f64,f64,f64,u32,u32,f64,f64
2023,50.830575,45.589601,54.832994,546,546,4.002419,5.240975
2024,60.220126,53.893155,64.519599,546,546,4.299473,6.326971
2025,55.327543,49.294262,59.831219,545,546,4.503676,6.033281
2026,52.815104,47.276055,57.113585,545,546,4.298482,5.539048


## 4. Temuan dan rekomendasi berbasis hasil

Angka di bawah dihitung otomatis agar tetap konsisten saat notebook dijalankan ulang.
Korelasi bersifat deskriptif dan tidak membuktikan sebab-akibat. Rasio penyerapan
tidak mengukur mutu layanan atau efisiensi belanja.

In [10]:
bt = trend.filter(pl.col('kategori') == 'Belanja').sort('tahun')
first, last = bt.row(0, named=True), bt.row(-1, named=True)
growth = 100 * (last['realisasi'] / first['realisasi'] - 1)
valid = annual_spend.filter(pl.col('rasio').is_not_null())
medians = valid.group_by('tahun').agg(pl.col('rasio').median().alias('median')).sort('tahun')
m_first, m_last = medians.row(0, named=True), medians.row(-1, named=True)
latest_corr = corr_all.filter(pl.col('tahun') == latest)['korelasi'][0]
latest_checked = corr_checked.filter(pl.col('tahun') == latest)['korelasi'][0]
budgets = trend.pivot(on='kategori', index='tahun', values='anggaran').sort('tahun')
budgets = budgets.with_columns((pl.col('Belanja') - pl.col('Pendapatan')).alias('selisih'))
b_last = budgets.row(-1, named=True)
repeated = audit_year.filter(pl.col('persen_pemda_identik') >= 50)['tahun'].to_list()
s_first, s_last = sept_summary.row(-2, named=True), sept_summary.row(-1, named=True)
extreme = annual_spend.filter(pl.col('tahun') == latest).sort('rasio', descending=True).row(0, named=True)
findings = f"""
1. **Tren nominal (grafik 1).** Pada panel tetap {len(panel_ids)} pemda,
   realisasi belanja berubah dari Rp {first['realisasi']/1e12:,.2f} triliun
   ({first['tahun']}) menjadi Rp {last['realisasi']/1e12:,.2f} triliun
   ({last['tahun']}), atau {growth:,.2f}%. Ini perubahan nominal tanpa koreksi
   inflasi. Agregat belum merupakan konsolidasi nasional dan sensitif terhadap pencilan.
2. **Perubahan penyerapan (grafik 2).** Median rasio belanja Desember pada panel
   tetap berubah dari {m_first['median']:.2f}% pada {m_first['tahun']} menjadi
   {m_last['median']:.2f}% pada {m_last['tahun']}, selisih
   {m_last['median']-m_first['median']:+.2f} poin persentase. Median memberi
   pembanding yang lebih tahan terhadap pencilan nominal dibanding total agregat.
3. **Sensitivitas korelasi antar-tahun (grafik 3).** Pada {latest}, korelasi
   anggaran–realisasi sebesar {latest_corr:.3f}; pada skenario rasio <=200%
   menjadi {latest_checked:.3f}. Perbedaan ini menunjukkan kepekaan terhadap nilai ekstrem.
   Rasio maksimum pada panel tahun tersebut {extreme['rasio']:,.2f}%; verifikasi
   laporan sumber sebelum menafsirkan lonjakan agregat sebagai perbaikan kinerja.
   Skenario sensitivitas tidak mengubah data utama dan bukan standar hukum.
4. **Perencanaan tahunan (grafik 4).** Pada {b_last['tahun']}, anggaran belanja
   Rp {b_last['Belanja']/1e12:,.2f} triliun dibanding anggaran pendapatan
   Rp {b_last['Pendapatan']/1e12:,.2f} triliun; selisih belanja dikurangi pendapatan
   Rp {b_last['selisih']/1e12:,.2f} triliun. Selisih ini perlu dibaca bersama akun
   pembiayaan; belum cukup untuk menyimpulkan kesehatan fiskal atau defisit terealisasi.
5. **Audit temporal (grafik 5).** Pada tahun {', '.join(map(str, repeated))},
   setidaknya separuh pemda panel bulanan memiliki realisasi identik pada 12 bulan.
   Pola tersebut mendukung pemeriksaan pengulangan snapshot, bukan klaim pola musiman.
   Jangan menjumlahkan bulan-bulan ini menjadi total tahunan.
6. **Perbandingan periode setara (grafik 6).** Pada panel September tetap
   {len(september_ids)} pemda, median penyerapan berubah dari {s_first['median']:.2f}%
   (September {s_first['tahun']}) menjadi {s_last['median']:.2f}%
   (September {s_last['tahun']}), selisih {s_last['median']-s_first['median']:+.2f}
   poin persentase. Gunakan perbandingan bulan yang sama untuk memantau tahun parsial;
   hasil September tidak boleh dibandingkan langsung dengan Desember sebagai kinerja
   tahunan. Telusuri ketepatan pelaporan serta pelaksanaan anggaran sebelum menetapkan tindakan.
"""
display(Markdown(findings))
(FIG_DIR / 'insights_eda.md').write_text(findings, encoding='utf-8')

Out[10]: 2164



1. **Tren nominal (grafik 1).** Pada panel tetap 498 pemda,
   realisasi belanja berubah dari Rp 502.63 triliun
   (2011) menjadi Rp 1,697.48 triliun
   (2025), atau 237.72%. Ini perubahan nominal tanpa koreksi
   inflasi. Agregat belum merupakan konsolidasi nasional dan sensitif terhadap pencilan.
2. **Perubahan penyerapan (grafik 2).** Median rasio belanja Desember pada panel
   tetap berubah dari 101.12% pada 2011 menjadi
   92.01% pada 2025, selisih
   -9.10 poin persentase. Median memberi
   pembanding yang lebih tahan terhadap pencilan nominal dibanding total agregat.
3. **Sensitivitas korelasi antar-tahun (grafik 3).** Pada 2025, korelasi
   anggaran–realisasi sebesar 0.262; pada skenario rasio <=200%
   menjadi 0.996. Perbedaan ini menunjukkan kepekaan terhadap nilai ekstrem.
   Rasio maksimum pada panel tahun tersebut 4,742.14%; verifikasi
   laporan sumber sebelum menafsirkan lonjakan agregat sebagai perbaikan kinerja.
   Skenario sensitivitas tidak mengubah data utama dan bukan standar hukum.
4. **Perencanaan tahunan (grafik 4).** Pada 2025, anggaran belanja
   Rp 1,339.62 triliun dibanding anggaran pendapatan
   Rp 1,279.48 triliun; selisih belanja dikurangi pendapatan
   Rp 60.14 triliun. Selisih ini perlu dibaca bersama akun
   pembiayaan; belum cukup untuk menyimpulkan kesehatan fiskal atau defisit terealisasi.
5. **Audit temporal (grafik 5).** Pada tahun 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022,
   setidaknya separuh pemda panel bulanan memiliki realisasi identik pada 12 bulan.
   Pola tersebut mendukung pemeriksaan pengulangan snapshot, bukan klaim pola musiman.
   Jangan menjumlahkan bulan-bulan ini menjadi total tahunan.
6. **Perbandingan periode setara (grafik 6).** Pada panel September tetap
   546 pemda, median penyerapan berubah dari 55.33%
   (September 2025) menjadi 52.82%
   (September 2026), selisih -2.51
   poin persentase. Gunakan perbandingan bulan yang sama untuk memantau tahun parsial;
   hasil September tidak boleh dibandingkan langsung dengan Desember sebagai kinerja
   tahunan. Telusuri ketepatan pelaporan serta pelaksanaan anggaran sebelum menetapkan tindakan.
